In [ ]:
!ls ../data

[1] Elliptic, www.elliptic.co.


[2] M. Weber, G. Domeniconi, J. Chen, D. K. I. Weidele, C. Bellei, T. Robinson, C. E. Leiserson, "Anti-Money Laundering in Bitcoin: Experimenting with Graph Convolutional Networks for Financial Forensics", KDD ’19 Workshop on Anomaly Detection in Finance, August 2019, Anchorage, AK, USA.


In [ ]:
import pandas as pd

DATA_DIR = "../data"

features = pd.read_csv(f"{DATA_DIR}/elliptic_txs_features.csv", header=None)
classes = pd.read_csv(f"{DATA_DIR}/elliptic_txs_classes.csv")
edges = pd.read_csv(f"{DATA_DIR}/elliptic_txs_edgelist.csv")

print("Features:", features.shape)
print("Classes:", classes.shape)
print("Edges:", edges.shape)

In [ ]:
display(features.head())
display(classes.head())
display(edges.head())

print("\nFeatures dtypes:")
print(features.dtypes)

print("\nClasses:")
print(classes["class"].value_counts(dropna=False))

print("\nMissing values:")
print("features:", features.isna().sum().sum())
print("classes:", classes.isna().sum().sum())
print("edges:", edges.isna().sum().sum())

In [ ]:
# COL 1 IS TIMESTEP, in 1 i have soime nodes iwth some edges
# i have 49 of these

In [ ]:
print("Number of timesteps:", features.iloc[:, 1].nunique())
print("Timesteps:", sorted(features.iloc[:, 1].unique()))

print("\nTransactions per timestep:")
print(features.iloc[:, 1].value_counts().sort_index())

At a high level:

203,769 transactions → nodes
234,355 edges → relationships between transactions
166 feature columns + timestep → node attributes
49 timesteps → temporal dimension
classes → labels for transactions, including unknown

So we already have three modeling views of the same data:

```
Elliptic dataset
├── Tabular:    transaction features → XGBoost
├── Graph:      transactions + edges + features → GCNN
└── Temporal:   transactions ordered by timestep → temporal model
```

undertand ok....


tab -> only consider the features of a transaction (features of a node)

yes, but let me understand first, for ex, for tabular i only consider the features of a node and make a model, for graphs i add edges so i can see relations, and for temporal i add the temproal component?

For the same transaction/node:

Tabular: use its own features → x_i → prediction.
Graph: use its features plus neighboring nodes/edges → prediction can incorporate relationships.
Temporal: use its features plus information across time → prediction can incorporate temporal patterns.

Conceptually:

                    What does the model see?


``` 
Tabular       node i ───────────────→ prediction

Graph         neighbors
                ↘  ↓  ↙
               node i ──────────────→ prediction

Temporal      t-2 → t-1 → node i(t) → t+1 ...
                         └────────────→ prediction

```  

One important nuance for later: "adding temporal component" doesn't necessarily mean simply adding the timestep as another feature. A temporal model usually means defining a sequence/order and allowing the model to learn dependencies across observations or timesteps.

And similarly, a GNN doesn't merely get an "edge feature"; the graph structure determines message passing between connected nodes.

So we'll progressively ask:

What additional structure are we giving the model, and what assumption does that introduce?

That's exactly the progression you want.

In [ ]:
print(classes["class"].value_counts(dropna=False))
print()
print(features.iloc[:, :10].head())
print()
print(features.iloc[:, 0].nunique())
print(edges.head())

## Understabd features

We're trying to establish:

Which column identifies the transaction.
Which column is the timestep.
Which columns are actual node features.
How many transactions are unknown, 1, and 2.

Once we know that, we'll make our first explicit modeling assumption: what information a simple tabular model is allowed to use.

In [ ]:
# understanbd features!!!!!!!!!!
print("Shape:", features.shape)

print("\nFirst 5 rows, first 10 columns:")
display(features.iloc[:5, :10])

print("\nColumn ranges:")
print("ID:", features.iloc[:, 0].min(), "→", features.iloc[:, 0].max())
print("Timestep:", features.iloc[:, 1].min(), "→", features.iloc[:, 1].max())

print("\nClasses:")
print(classes["class"].value_counts(dropna=False))

What we know
203,769 transactions
Column 0 = transaction ID
Column 1 = timestep
Columns 2:167 = 165 transaction features
49 timesteps
Labels:
1: 4,545
2: 42,019
unknown: 157,205

So the labeled subset is only 46,564 transactions, while most transactions are unknown.

The first modeling implication is important:

For the initial supervised experiments, we should probably use only transactions with known labels.

And we should not treat unknown as a third class automatically. In this dataset, it represents unlabeled transactions rather than necessarily a legitimate/negative class.

In [ ]:
print("\nFeature summary:")
display(features.iloc[:, 2:].describe().T.head(20))

In [ ]:
# rel. bween timesteps and labels; important for temporal models

In [ ]:
df = features.iloc[:, :2].copy()
df.columns = ["tx_id", "timestep"]

df["class"] = classes["class"]

display(
    df.groupby(["timestep", "class"])
      .size()
      .unstack(fill_value=0)
)

Good. One clear observation: the class distribution changes substantially across timesteps, and there are many unknown transactions throughout the timeline.

# Model 1 : XGBoost

Our initial assumption will be deliberately simple:

Each transaction is an independent tabular observation. We use only its 165 features and ignore the graph edges and temporal relationships.

So:

```
transaction
   ↓
165 features
   ↓
XGBoost
   ↓
class prediction
```  


We should also exclude transaction ID and timestep initially. The ID has no meaningful predictive interpretation, and excluding timestep lets us establish a clean feature-only baseline.

Before training, we'll need to decide the train/validation split. Since this is a temporal dataset, that's actually an important assumption—we shouldn't blindly use a random split. Let's handle that nex

For this first probe, I'd use a temporal split: train on earlier timesteps and validate on later ones. This avoids letting future transactions leak into training.

**Assumption**

We're assuming:

The model will be trained on past transactions and applied to future transactions.

That's a much more meaningful first setup for this dataset than randomly mixing transactions from all 49 timesteps.

One caveat: 34/49 is just a practical probe, not a claim that this is the correct production split. We'll keep it simple for now.

Then we can train the XGBoost model

In [ ]:
# Known-label transactions only
mask = classes["class"] != "unknown"

X = features.loc[mask, 2:].values
y = classes.loc[mask, "class"].map({"1": 0, "2": 1}).values
t = features.loc[mask, 1].values

# Train: timesteps 1–34
# Validation: timesteps 35–49
train_mask = t <= 34
val_mask = t > 34

X_train, y_train = X[train_mask], y[train_mask]
X_val, y_val = X[val_mask], y[val_mask]

print("Train:", X_train.shape)
print("Validation:", X_val.shape)

print("\nTrain classes:")
print(pd.Series(y_train).value_counts())

print("\nValidation classes:")
print(pd.Series(y_val).value_counts())

In [ ]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    random_state=42,
)

model.fit(X_train, y_train)

val_proba = model.predict_proba(X_val)[:, 1]

In [ ]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report,
)

print("ROC-AUC:", roc_auc_score(y_val, val_proba))
print("PR-AUC:", average_precision_score(y_val, val_proba))

print("\nClassification report @ 0.5:")
print(classification_report(y_val, val_proba >= 0.5))

For this first experiment, don't tune anything yet.

We're simply answering:

Can a standard tabular model learn useful information from the 165 transaction features when trained on earlier timesteps and evaluated on later ones?

Give me the output and we'll interpret it before introducing the graph.

One important point: don't be distracted by the 98% accuracy. The validation set is highly imbalanced (1083 vs 15587), so PR-AUC and minority-class recall/precision are more informative.

For our purposes, we can record this as:

XGBoost: using only the 165 transaction features, with a temporal train/validation split, the model learns substantial predictive signal.

# Graph Model

Now we'll change one thing:

Instead of treating transactions independently, we'll allow the model to use the edges between transactions.

The basic setup becomes:

Node features X
      +
Edges
      ↓
   GCNN/GNN
      ↓
transaction class

Before choosing the exact GNN architecture, let's inspect the edges and establish one assumption: are the transaction IDs in the edge list the same IDs as column 0 of the features?


This let us construct the graph clearly instead make assumptions about the identifiers.

In [ ]:
print(edges.head())
print(edges.columns)

print("Unique source nodes:", edges.iloc[:, 0].nunique())
print("Unique destination nodes:", edges.iloc[:, 1].nunique())

print("Edges:", len(edges))

print(
    "Feature IDs appearing in edges:",
    edges.iloc[:, 0].isin(features.iloc[:, 0]).mean(),
    edges.iloc[:, 1].isin(features.iloc[:, 0]).mean()
)

Perfect. So the graph construction is straightforward:

203,769 nodes = transactions
234,355 edges
txId1 → txId2 IDs map exactly to the feature table
Every endpoint in the edge list exists in the feature data.

One assumption to make explicit: we'll initially treat the edges as an undirected graph for a standard GCN. We're deliberately ignoring transaction direction for this first probe.

Next step: build the graph

We'll use PyTorch Geometric if you have it installed:

In [ ]:
import torch
from torch_geometric.data import Data

# Node features: exclude txId and timestep
X_graph = torch.tensor(
    features.iloc[:, 2:].values,
    dtype=torch.float32
)

# Map transaction ID -> node index
tx_to_idx = {
    tx_id: i
    for i, tx_id in enumerate(features.iloc[:, 0])
}

src = edges["txId1"].map(tx_to_idx).values
dst = edges["txId2"].map(tx_to_idx).values

edge_index = torch.tensor(
    [src, dst],
    dtype=torch.long
)

# Make graph undirected
edge_index = torch.cat(
    [edge_index, edge_index.flip(0)],
    dim=1
)

print("X:", X_graph.shape)
print("edge_index:", edge_index.shape)

In [ ]:
# attach labels

labels = classes["class"].map({
    "1": 0,
    "2": 1,
    "unknown": -1
}).values

y_graph = torch.tensor(labels, dtype=torch.long)

print("Labeled nodes:", (y_graph >= 0).sum().item())
print("Unknown nodes:", (y_graph == -1).sum().item())

Important: unlike XGBoost, the GCN will receive the graph containing all nodes, including unknown-label nodes. The unknown nodes simply won't contribute to the supervised loss.

Next we'll create temporal train/validation masks on the nodes, preserving the same 1–34 / 35–49 split we used for XGBoost.

In [ ]:
timesteps = features.iloc[:, 1].values

train_mask = (
    (timesteps <= 34) &
    (y_graph.numpy() >= 0)
)

val_mask = (
    (timesteps > 34) &
    (y_graph.numpy() >= 0)
)

train_mask = torch.tensor(train_mask, dtype=torch.bool)
val_mask = torch.tensor(val_mask, dtype=torch.bool)

print("Train nodes:", train_mask.sum().item())
print("Validation nodes:", val_mask.sum().item())

In [ ]:
data = Data(
    x=X_graph,
    edge_index=edge_index,
    y=y_graph,
    train_mask=train_mask,
    val_mask=val_mask,
)

print(data)

One subtlety

Because this is a single graph containing all timesteps, the GCN can technically pass messages through edges involving future nodes during validation.

We're therefore making an important simplification:

For this first GNN probe, we use a temporal split for the labels/loss, but we do not yet enforce temporal isolation of the graph structure.

That's acceptable for a quick "does a GNN train?" experiment, but it means its validation result isn't directly equivalent to a strict future-only deployment scenario.

We can address that later if the graph experiment is interesting.

Now we can define a tiny 2-layer GCN and train it



.......IMPROVE LATER!!!!!!!!!!!!!!!!!!!!


ADD A PLOT


In [ ]:
# small GCN
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv


class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()

        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)

        return x

In [ ]:
model = GCN(
    in_channels=data.num_node_features,
    hidden_channels=64,
    out_channels=2,
)

In [ ]:
# train on the training mask
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01,
    weight_decay=5e-4,
)

criterion = torch.nn.CrossEntropyLoss()

for epoch in range(100):

    model.train()

    optimizer.zero_grad()

    logits = model(data.x, data.edge_index)

    loss = criterion(
        logits[data.train_mask],
        data.y[data.train_mask],
    )

    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch:03d} | Loss: {loss.item():.4f}")

The important conceptual point is that all nodes participate in the forward pass, but only labeled training nodes contribute to the supervised loss.

In [ ]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report,
)

model.eval()

with torch.no_grad():
    logits = model(data.x, data.edge_index)
    proba = torch.softmax(logits, dim=1)[:, 1]

y_val = data.y[data.val_mask].numpy()
proba_val = proba[data.val_mask].numpy()

print("ROC-AUC:", roc_auc_score(y_val, proba_val))
print("PR-AUC:", average_precision_score(y_val, proba_val))

pred_val = (proba_val >= 0.5).astype(int)

print("\nClassification report @ 0.5:")
print(classification_report(y_val, pred_val))

What you're testing

Your progression is now very clean:

Experiment	Information available to model
XGBoost	Transaction's own 165 features
GCN	Transaction features + graph neighbors
TCN	Transaction features + temporal structure

So if the GCN behaves differently from XGBoost, you have evidence that relational structure contains information beyond the node's own features.

One important caveat remains: this GCN uses the whole graph during message passing, including edges involving later timesteps. So don't interpret its validation score as a strict "train on past → predict future using only information available at that time" experiment yet.

For this exploratory repo, I'd actually keep that caveat documented and move on to the TCN rather than complicating the GCN no

That is a useful result. The GCN is clearly learning the task, but its behavior differs substantially from XGBoost.

XGBoost vs GCN
Model	Class 0 precision	Class 0 recall	Class 1 precision	Class 1 recall
XGBoost	0.94	0.72	0.98	1.00
GCN	0.82	0.32	0.95	1.00

The main difference is class 0 recall: it falls from 0.72 → 0.32.

So the GCN is predicting considerably more validation transactions as class 1. Its predictions are still quite accurate when it predicts class 0 (precision 0.82), but it misses most of the actual class-0 transactions.

Also, don't overinterpret the accuracy of 0.95: class 1 dominates the validation set (15,587 / 16,670 ≈ 93.5%), so predicting class 1 frequently already gives high accuracy.

What I'd do next

Before interpreting the GCN as better/worse, print ROC-AUC and PR-AUC, since those don't depend on the arbitrary 0.5 threshold

In [ ]:
print("ROC-AUC:", roc_auc_score(y_val, proba_val))
print("PR-AUC:", average_precision_score(y_val, proba_val))

The interesting question for this project isn't "which model wins?" but:

    Does adding graph structure change the predictive signal compared with treating each transaction independently?

Your current classification report already suggests that the answer is yes: the GCN has learned a different decision boundary from the purely tabular model.

-----------
Good. Now the comparison is complete:

Model	ROC-AUC	PR-AUC
XGBoost	0.9365	0.9947
GCN	0.8682	0.9868

A few observations are enough for this exploratory project:

Both models learn substantial signal. Neither is close to random ranking.
XGBoost has higher ROC-AUC and PR-AUC on this validation split.
The difference is larger in ROC-AUC: 0.9365 vs 0.8682.
The PR-AUC difference is smaller: 0.9947 vs 0.9868.
The GCN's threshold behavior is noticeably different: class-0 recall was only 0.32, despite its high PR-AUC.
Because class 1 represents ~93.5% of this validation set, the PR-AUC baseline is already high (~0.935). So the absolute PR-AUC values shouldn't be interpreted without that context.

Most importantly, I would not conclude that graph structure is useless from this. Your experiment changed the modeling assumptions, and the GCN successfully incorporated the graph. It simply didn't improve the metric in this particular setup.

For the repo, I'd record something like:

GCN: A simple 2-layer GCN was trained using the transaction graph and node features. It achieved ROC-AUC 0.868 and PR-AUC 0.987 on the temporal validation split. Compared with the tabular XGBoost model (ROC-AUC 0.936, PR-AUC 0.995), the GCN performed lower on both ranking metrics in this initial configuration. This experiment demonstrates the use of graph structure rather than establishing that a GNN is preferable for this dataset.

Then stop here with the GCN. Don't tune it.

Next is the interesting part: TCN, where we introduce temporal structure as a new modeling assumption.


<font color='red'> see clearly time assumtions!!!!!! </font>

## TEMPORAL STRUCTURE

Use the graph to construct temporal neighborhoods:

transaction i
     │
     ├── timestep t-2 neighbors
     ├── timestep t-1 neighbors
     ├── timestep t   ← target
     └── timestep t+1 neighbors

But that's starting to become a fairly specialized temporal-GNN rather than a simple TCN.

So for your stated goal—probe XGBoost → GCN → TCN—I'd make one explicit simplification:

Treat the dataset rows as an ordered temporal stream and construct fixed-size windows over that ordering.

This is not claiming that row order represents a meaningful transaction sequence. It is an exploratory temporal probe, and that assumption should be documented.

In [ ]:
# dont icnlkude TS as feat
import numpy as np

X = features.iloc[:, 2:].values.astype(np.float32)
y = classes["class"].map({
    "1": 0,
    "2": 1,
    "unknown": -1,
}).values

timesteps = features.iloc[:, 1].values

In [ ]:
window_size = 10

X_windows = []
y_windows = []
time_windows = []

for i in range(len(X) - window_size + 1):

    X_windows.append(
        X[i:i + window_size]
    )

    y_windows.append(
        y[i:i + window_size]
    )

    time_windows.append(
        timesteps[i:i + window_size]
    )

X_windows = np.array(X_windows)
y_windows = np.array(y_windows)
time_windows = np.array(time_windows)

print(X_windows.shape)

In [ ]:
"""
For a supervised exploratory experiment, use:

A window is labeled by the last transaction in the window.

This avoids propagating an anomaly label backward across the whole window.
"""

In [ ]:
window_labels = y_windows[:, -1]
window_times = time_windows[:, -1]

In [ ]:
# keep only label targets
known = window_labels >= 0

X_windows = X_windows[known]
window_labels = window_labels[known]
window_times = window_times[known]

In [ ]:
train_mask = window_times <= 34
val_mask = window_times > 34

X_train = X_windows[train_mask]
y_train = window_labels[train_mask]

X_val = X_windows[val_mask]
y_val = window_labels[val_mask]

But there's a major caveat

This window construction assumes that adjacent rows are temporally meaningful.

If the CSV is grouped by timestep, adjacent rows may simply be arbitrary transactions from the same timestep rather than a meaningful sequence. In that case the TCN is effectively learning patterns in row ordering, not transaction temporal dynamics.

So before writing the TCN, check:

In [ ]:
print(time_windows[:20])

print(
    np.unique(
        np.diff(timesteps[:1000]),
        return_counts=True
    )
)

WITH THIS TECHNIQUE:

TCN: Not implemented in the initial exploration. Although the dataset provides a timestep feature, transactions within each timestep are not ordered observations forming a natural sequence. Constructing arbitrary windows from the CSV row order would therefore introduce an unjustified temporal assumption. A meaningful temporal model would require defining an appropriate temporal entity/sequence representation first.

--
ALKTERNATIVES
To build a valid temporal model on Elliptic without making false row-order assumptions, choose one of these sequence representations:Snapshot-Level Sequence (Macro-level): Treat each timestep $t \in \{1, \dots, 49\}$ as an atomic graph snapshot $G_t$. You then feed the sequence of graphs $(G_1, G_2, \dots, G_t)$ into a temporal architecture (like EvolveGCN or GNN-LSTM). The sequence unit is the entire snapshot, not individual CSV rows.Graph Topology & Random Walks (Structural sequence): Sequence models like Node2Vec or CTDNE build sequences by taking random walks along the transaction directed edges. The order is derived from who paid whom (the graph structure), not the CSV row index.

1. Snapshot-level sequence — valid temporal modeling

This is the most natural temporal interpretation of the Elliptic dataset.

You have:

t=1 → G₁
t=2 → G₂
t=3 → G₃
...
t=49 → G₄₉

Each G_t contains the transactions/nodes and edges associated with timestep t.

The temporal unit is therefore:

one timestep = one graph snapshot

rather than:

one row = one timestep

Then a temporal GNN could learn:

G₁ → G₂ → G₃ → ... → G_t

For example:

          G₁ ──→ G₂ ──→ G₃ ──→ ... ──→ G_t
           │      │      │                │
          GNN    GNN    GNN              GNN
           │      │      │                │
           └──────┴──────┴──→ temporal ──┘
                              model

But this is no longer a simple TCN. You would first need a graph representation at each timestep and then a mechanism to turn each graph into a representation that a temporal model can consume.

A relatively simple version could be:

G_t
 ↓
GCN
 ↓
graph-level embedding h_t
 ↓
[h₁, h₂, ..., h_t]
 ↓
TCN / LSTM
 ↓
prediction

That would actually be a GCN + TCN, rather than a pure TCN.

And there is another question: your prediction target is a transaction, while a graph-level temporal embedding is naturally a snapshot-level representation. You'd need to design how that temporal context is attached back to individual transactions.

So this is valid, but considerably more involved.

# Snapshot level sequence

. The guideline should drive the exploration, not constrain it. Option 1 is a much more meaningful temporal experiment for Elliptic.

Let's build the simplest useful version:

Each timestep is a graph snapshot → GCN produces node embeddings → temporal model consumes each node's history.

There is one important detail: because our target is still transaction-level classification, we don't want to collapse each graph into one graph-level embedding. We want to preserve each node's identity across time.

Representation

For each timestep:

G₁              G₂              G₃                    G₄₉
│               │               │                       │
GCN             GCN             GCN                    GCN
│               │               │                       │
H₁              H₂              H₃                    H₄₉

where:

H_t = [h₁,t, h₂,t, ..., h_N,t]

Then for a transaction/node that exists across several timesteps:

h_i,1 → h_i,2 → h_i,3 → ... → h_i,t
                 │
                TCN
                 │
            class prediction

That is a genuine temporal model.

But first: understand the temporal graph

Before coding the TCN, let's inspect whether nodes actually persist across timesteps.

We already know there are 49 snapshots. Now calculate:

In [ ]:
tx_id = features.iloc[:, 0].values
t = features.iloc[:, 1].values

df_tmp = pd.DataFrame({
    "txId": tx_id,
    "timestep": t,
})

# Number of unique timesteps for each transaction
node_lifetimes = df_tmp.groupby("txId")["timestep"].nunique()

print(node_lifetimes.describe())
print(
    "Nodes appearing in >1 timestep:",
    (node_lifetimes > 1).sum()
)

Right. This is an important discovery.

Every transaction/node exists in exactly one timestep. Therefore, we cannot construct:

transaction i:
h_i,t-2 → h_i,t-1 → h_i,t

because there is no persistent i across timesteps.

But this does not mean the dataset has no temporal structure. It means the temporal structure is at the graph/snapshot level, not at the persistent-node level.

So let's change the temporal formulation

We have:

G₁, G₂, G₃, ..., G₄₉

Each graph contains a different set of transaction nodes.

We can therefore create a snapshot-level temporal representation:

G₁ ──→ G₂ ──→ G₃ ──→ ... ──→ G₄₉
│       │       │               │
GCN     GCN     GCN             GCN
│       │       │               │
z₁      z₂      z₃              z₄₉

where z_t summarizes the graph at timestep t.

Then:

z₁ → z₂ → z₃ → ... → z₄₉
                 │
                TCN
                 │
        temporal representation

But there's a major question:

What should the TCN predict?

If we use a graph-level z_t, the natural target becomes something about timestep t, such as:

number/proportion of illicit transactions,
distribution of node features,
graph statistics,
whether a timestep contains unusual activity.

That's no longer transaction-level classification.

If we want to retain transaction-level classification, we need to exploit cross-snapshot edges.

And here's where the Elliptic graph becomes interesting.

Inspect the edges across timesteps

Let's determine how often an edge connects transactions from different timesteps.

In [ ]:
tx_to_time = dict(
    zip(
        features.iloc[:, 0],
        features.iloc[:, 1]
    )
)

edges["t1"] = edges["txId1"].map(tx_to_time)
edges["t2"] = edges["txId2"].map(tx_to_time)

print(
    "Same timestep:",
    (edges["t1"] == edges["t2"]).mean()
)

print(
    "Different timestep:",
    (edges["t1"] != edges["t2"]).mean()
)

In [ ]:
temporal_distance = (
    edges["t1"] - edges["t2"]
).abs()

print(temporal_distance.describe())

Excellent. That gives us the actual structure of this dataset:

Each transaction/node belongs to exactly one timestep.
Every edge connects nodes within the same timestep.
Therefore there are 49 disconnected temporal graph snapshots.
There is no node-level persistence and no cross-time graph connectivity.

So we should not manufacture temporal relationships between transactions.

But your snapshot-level idea remains completely valid. In fact, this makes the modeling formulation cleaner:

G₁    G₂    G₃    ...    G₄₉
 │     │     │            │
 GNN   GNN   GNN          GNN
 │     │     │            │
 z₁    z₂    z₃    ...    z₄₉
  \     \     \          /
       temporal model
            │
      temporal output

The question becomes what z_t and the target should be.


A natural temporal task

Since every timestep is a graph snapshot, we can ask:

Can the model use previous snapshots to predict properties of the next snapshot?

For example, predict the class composition of the next timestep.

But if your objective is specifically anomaly/illicit transaction classification, this changes the task substantially from XGBoost and GCN.

That's not necessarily bad. It is actually a useful distinction:

Model	Unit	Structure exploited
XGBoost	transaction	features
GCN	transaction	features + same-snapshot graph
Temporal GNN	snapshot / sequence	evolution across snapshots

The models would no longer be apples-to-apples, but they would answer different questions about the dataset.


One more possibility worth considering

Because there are only 49 snapshots, we could create a sequence of snapshot-level statistics/features and feed that into a TCN:

t=1  →  [snapshot statistics]
t=2  →  [snapshot statistics]
...
t=49 →  [snapshot statistics]

For example:

z_t = [
    number of transactions,
    number of edges,
    graph density,
    mean feature_1,
    mean feature_2,
    ...
]

Then:

[z₁, z₂, ..., z₄₉] → TCN → temporal prediction

This is much simpler than EvolveGCN and gives you a genuine TCN experiment.

However, it answers a different question: temporal evolution of the dataset, rather than transaction classificatio

For this project, I'd actually keep both ideas conceptually separate

Your exploration can now say:

Tabular transaction modeling
    └── XGBoost

Within-snapshot relational modeling
    └── GCN

Across-snapshot temporal modeling
    └── TCN / temporal GNN

That's a stronger exploration than forcing all three models onto exactly the same prediction formulation.

Next I'd inspect the 49 snapshots themselves—number of nodes, edges, class distribution, and a few aggregate feature statistics. That tells us whether there is enough temporal variation for a snapshot-level TCN to learn anything meaningful before we build it

## better formulation
can i establish the problem liket his:volutionary / Dynamic GNN*"Given how laundering structures have shifted from step $t-5$ to $t-1$, is the current structure surrounding node $X$ at time $t$ the new pattern for illicit activity?"*EvolveGCN, GNN-LSTM

Yes. That is actually a better formulation of the temporal problem than trying to make a TCN consume arbitrary transaction sequences.

The key is that the temporal object is not a persistent node. It is the evolution of graph structure across snapshots.

Your question:

Given how laundering structures have shifted from step \(t-5\) to \(t-1\), is the current structure surrounding node \(X\) at time \(t\) the new pattern for illicit activity?

is a legitimate dynamic/evolutionary GNN problem.

The formulation

You have:

G₁ → G₂ → G₃ → ... → Gₜ₋₅ → ... → Gₜ₋₁ → Gₜ

Each \(G_t\) is a different graph. Nodes don't persist, but graph structures evolve.

The model can therefore learn something like:

$$ G_{t-k:t-1} \rightarrow \text{representation of the evolving graph} \rightarrow G_t $$

and use that temporal context when classifying nodes in \(G_t\).

Conceptually:

             Historical graph evolution
        ┌──────────────────────────────┐
        │                              │
       Gₜ₋₅ → Gₜ₋₄ → ... → Gₜ₋₂ → Gₜ₋₁
        │       │              │        │
       GNN     GNN            GNN      GNN
        │       │              │        │
        └───────┴──────────────┴────────┘
                         │
                  temporal mechanism
                    (LSTM / GRU)
                         │
                         ▼
                        Gₜ
                         │
                        GNN
                         │
                         ▼
                 node classification

This is much closer to what EvolveGCN / GNN-LSTM-style approaches are designed to address.

One important correction

I would avoid saying:

"Given how laundering structures have shifted..."

as though we've already established that the structures represent laundering behavior.

At the problem-definition stage, something more rigorous is:

"Given how graph structures have evolved from previous timesteps, does the temporal evolution of a node's structural context provide information for classifying transactions at the current timestep?"

Then the illicit activity interpretation comes from the labels.

That gives you a clean hypothesis:

Temporal hypothesis: the graph structure surrounding transactions evolves over time, and historical graph structure may provide predictive information about the class of transactions in the current snapshot.

And this changes what I'd recommend

I would not build a TCN over aggregated snapshot statistics as the main temporal experiment. That's technically valid, but it throws away the transaction-level graph structure that makes Elliptic interesting.

Instead, your exploration can now be:

1. XGBoost
   "Are node features sufficient?"

2. GCN
   "Does the current transaction graph provide additional information?"

3. Dynamic GNN
   "Does the evolution of graph structure across previous
    timesteps provide additional information?"

That's a much more coherent progression.

And importantly, the fact that nodes don't persist isn't a blocker. We're not asking whether the same transaction changes over time. We're asking whether the population-level graph structure evolves in a way that helps classify the next snapshot's transactions.

The next thing I'd investigate is therefore how to define the temporal neighborhood from \(t-k,\ldots,t-1\) for a node in \(G_t\) using the graph topology, without inventing nonexistent cross-time edges. That is the central representation problem before choosing between EvolveGCN and GNN-LST

### GNN-LSTM

Yes. Given what we discovered, I would implement the GNN-LSTM version first, rather than jumping directly into EvolveGCN internals.

The important design decision is:

Since nodes do not persist and there are no cross-timestep edges, we cannot define a historical neighborhood for node \(X\). Instead, we let the GNN encode each snapshot and the LSTM encode the evolution of the graph, then use that temporal state when classifying nodes in the current snapshot.

So the model becomes:

G₁ ── GCN ──► H₁ ── pool ──► z₁ ──┐
G₂ ── GCN ──► H₂ ── pool ──► z₂ ──┤
G₃ ── GCN ──► H₃ ── pool ──► z₃ ──┤──► LSTM ──► hₜ
...                                 │
Gₜ₋₁ ─ GCN ─► Hₜ₋₁ ─ pool ─► zₜ₋₁ ─┘
                                      │
Gₜ ─────── GCN ───────► Hₜ ──────────┤
                                      ▼
                              [Hₜ, hₜ]
                                      │
                                 classifier
                                      │
                              transaction labels

This gives you a very clean temporal hypothesis:

$$ P(y_i^t \mid X_i^t,G_t,G_{t-1},...,G_{t-k}) $$

rather than pretending that transaction \(i\) itself exists throughout time.

In [ ]:
# split the graph into snapshjots
from torch_geometric.data import Data

snapshots = []

for t in range(1, 50):

    node_mask = timesteps == t

    global_nodes = np.where(node_mask)[0]

    # Global node index -> local snapshot index
    global_to_local = {
        node: i
        for i, node in enumerate(global_nodes)
    }

    # Keep only edges whose two endpoints belong to this timestep
    edge_mask = (
        node_mask[edge_index[0].numpy()]
        & node_mask[edge_index[1].numpy()]
    )

    edges_t = edge_index[:, edge_mask]

    # Convert global indices to local snapshot indices
    local_src = torch.tensor(
        [global_to_local[i.item()] for i in edges_t[0]],
        dtype=torch.long,
    )

    local_dst = torch.tensor(
        [global_to_local[i.item()] for i in edges_t[1]],
        dtype=torch.long,
    )

    edge_index_t = torch.stack(
        [local_src, local_dst]
    )

    snapshot = Data(
        x=X_graph[node_mask],
        edge_index=edge_index_t,
        y=y_graph[node_mask],
    )

    snapshots.append(snapshot)

    print(
        f"t={t:02d} | "
        f"nodes={snapshot.num_nodes} | "
        f"edges={snapshot.num_edges}"
    )

In [ ]:
class GCNEncoder(torch.nn.Module):

    def __init__(self, in_channels, hidden_channels):
        super().__init__()

        self.conv1 = GCNConv(
            in_channels,
            hidden_channels,
        )

        self.conv2 = GCNConv(
            hidden_channels,
            hidden_channels,
        )

    def forward(self, x, edge_index):

        x = self.conv1(x, edge_index)
        x = F.relu(x)

        x = self.conv2(x, edge_index)

        return x
    
"""
The important difference from the previous experiment is that this GCN is now applied independently to every snapshot.
"""

3. Add the temporal component

We need one vector representing each graph snapshot.

The simplest choice is mean pooling:

z
t
	​

=
∣V
t
	​

∣
1
	​

i∈V
t
	​

∑
	​


Since each snapshot is one graph, we can simply do:

z_t = h_t.mean(dim=0)

Then the LSTM receives:

[z₁, z₂, z₃, ..., zₜ₋₁]

and produces the historical temporal state.

In [ ]:
from torch_geometric.nn import global_mean_pool
z_t = h_t.mean(dim=0)

In [ ]:
class GNNLSTM(torch.nn.Module):

    def __init__(
        self,
        in_channels,
        hidden_channels,
        temporal_hidden,
        num_classes=2,
    ):
        super().__init__()

        self.gnn = GCNEncoder(
            in_channels,
            hidden_channels,
        )

        self.lstm = torch.nn.LSTM(
            input_size=hidden_channels,
            hidden_size=temporal_hidden,
            batch_first=True,
        )

        self.classifier = torch.nn.Linear(
            hidden_channels + temporal_hidden,
            num_classes,
        )

    def forward(self, history, current):

        snapshot_embeddings = []

        # Encode historical snapshots
        for graph in history:

            h = self.gnn(
                graph.x,
                graph.edge_index,
            )

            z = h.mean(dim=0)

            snapshot_embeddings.append(z)

        if snapshot_embeddings:

            sequence = torch.stack(
                snapshot_embeddings
            ).unsqueeze(0)

            _, (h_temporal, _) = self.lstm(
                sequence
            )

            temporal_context = h_temporal[-1]

        else:

            temporal_context = torch.zeros(
                self.lstm.hidden_size,
                device=current.x.device,
            )

        # Encode current snapshot
        h_current = self.gnn(
            current.x,
            current.edge_index,
        )

        # Same temporal context for every current node
        temporal_context = temporal_context.expand(
            current.num_nodes,
            -1,
        )

        combined = torch.cat(
            [
                h_current,
                temporal_context,
            ],
            dim=1,
        )

        logits = self.classifier(combined)

        return logits

5. The temporal training task

For example, use the previous 5 snapshots to classify the current snapshot:

t=6:

G₁ → G₂ → G₃ → G₄ → G₅
                         \
                          LSTM
                            \
                             G₆ → predictions

Then for t=7:

G₂ → G₃ → G₄ → G₅ → G₆
                         \
                          LSTM
                            \
                             G₇ → predictions

And so on.

That means we can train on timesteps 6–34 and validate on 35–49, preserving your original temporal split.

In [ ]:
# window
history_size = 5

model = GNNLSTM(
    in_channels=165,
    hidden_channels=64,
    temporal_hidden=32,
    num_classes=2,
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01,
    weight_decay=5e-4,
)

criterion = torch.nn.CrossEntropyLoss()

In [ ]:
# train
for epoch in range(50):

    model.train()

    total_loss = 0.0

    for t in range(history_size, 34):

        history = snapshots[
            t - history_size:t
        ]

        current = snapshots[t]

        known = current.y >= 0

        if known.sum() == 0:
            continue

        optimizer.zero_grad()

        logits = model(
            history,
            current,
        )

        loss = criterion(
            logits[known],
            current.y[known],
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if epoch % 10 == 0:

        print(
            f"Epoch {epoch:03d} | "
            f"Loss {total_loss:.4f}"
        )

There is an important indexing detail here: snapshots[0] is timestep 1, so t=34 corresponds to timestep 35. We should clean this up when we write the final experiment, but the conceptual split is:

TRAIN: t = 1 ... 34
VAL:   t = 35 ... 49

and the validation prediction at timestep \(t\) uses only:

t-5 ... t-1

as historical context.

7. Why this is actually a meaningful experiment

Now your three experiments represent three progressively different hypotheses:

XGBoost
$$ P(y_i \mid x_i) $$

Does the transaction's own feature vector contain enough information?

GCN
$$ P(y_i \mid x_i,G_t) $$

Does the transaction's current graph neighborhood provide additional information?

GNN-LSTM
$$ P(y_i^t \mid x_i^t,G_t,G_{t-1},...,G_{t-5}) $$

Does the evolution of the graph provide additional information about transactions in the current snapshot?

That's a genuinely interesting progression.

And there's no need to pretend that the same node exists across time. The LSTM is modeling graph evolution, while the GCN handles the current transaction-level structure.

One caveat: in this first implementation, the temporal context is global to the snapshot, because mean pooling compresses the historical graph into one vector. That's a deliberate simplification. If this produces an interesting result, the next level would be to make the temporal context more structurally localized—potentially moving toward an actual EvolveGCN-style architecture.

In [ ]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
)
import torch

model.eval()

y_true = []
y_proba = []
y_pred = []

history_size = 5

with torch.no_grad():
    for t in range(34, len(snapshots)):  # timesteps 35–49
        history = snapshots[t-history_size:t]
        current = snapshots[t]

        known = current.y >= 0

        if known.sum() == 0:
            continue

        logits = model(history, current)
        proba = torch.softmax(logits, dim=1)[:, 1]
        pred = logits.argmax(dim=1)

        y_true.extend(current.y[known].cpu().numpy())
        y_proba.extend(proba[known].cpu().numpy())
        y_pred.extend(pred[known].cpu().numpy())


# Metrics
roc_auc = roc_auc_score(y_true, y_proba)
pr_auc = average_precision_score(y_true, y_proba)

print(f"ROC-AUC: {roc_auc:.4f}")
print(f"PR-AUC:  {pr_auc:.4f}")

print("\nClassification report:")
print(
    classification_report(
        y_true,
        y_pred,
        digits=4,
    )
)

print("Confusion matrix:")
print(confusion_matrix(y_true, y_pred))


"""
One important detail: this evaluates the exact temporal setup you trained: each validation timestep gets its preceding
5 snapshots as history, with no validation labels used as historical input."""

A few observations:

ROC-AUC: GNN-LSTM is slightly below the GCN.
PR-AUC: also slightly below the GCN.
Class 1: recall is essentially perfect, as with the other models.
Class 0: this is where the model struggles. It only identifies 219/1083 class-0 transactions at the default 0.5 threshold.
The high accuracy (0.9457) is not particularly informative here because validation is heavily dominated by class 1 (~93.5%).

The important methodological point is that this does not establish that temporal information is useless. It establishes that this particular temporal representation—mean-pooling each historical graph into a single vector and feeding those vectors to an LSTM—didn't outperform the simpler GCN.

And that's actually a useful result for your exploration.

I would record the experiment roughly as:

GNN-LSTM: modeled temporal evolution as a sequence of graph-level representations obtained by mean-pooling GCN node embeddings from the previous five snapshots. Performance was slightly below the static GCN, suggesting that this simple global snapshot representation did not provide additional predictive signal over the current graph structure.

I would not tune it yet. The interesting next question is whether you want to test a more structurally appropriate temporal GNN such as EvolveGCN, rather than trying to squeeze performance out of this GNN-LSTM.

## EvolveGCN

Yes. EvolveGCN is a better next experiment because it changes the temporal mechanism rather than just tuning the LSTM.

The key idea is:

G₁ ── GCN(W₁) ──► H₁
        ▲
        │ evolve
G₂ ── GCN(W₂) ──► H₂
        ▲
        │ evolve
G₃ ── GCN(W₃) ──► H₃
        ...

Instead of creating a global temporal vector with pooling, the GCN parameters evolve from one snapshot to the next. This is particularly appropriate here because your nodes do not persist across timesteps.

For your first implementation, I'd use the simpler EvolveGCN-H formulation: an LSTM evolves the GCN weight matrices over time

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv


class EvolveGCN(nn.Module):

    def __init__(
        self,
        in_channels,
        hidden_channels,
        num_classes=2,
    ):
        super().__init__()

        self.hidden_channels = hidden_channels

        # Initial GCN weights
        self.weight1 = nn.Parameter(
            torch.randn(in_channels, hidden_channels)
        )

        self.weight2 = nn.Parameter(
            torch.randn(hidden_channels, hidden_channels)
        )

        # LSTMs evolve the GCN weights
        self.evolve1 = nn.LSTMCell(
            input_size=in_channels,
            hidden_size=in_channels * hidden_channels,
        )

        self.evolve2 = nn.LSTMCell(
            input_size=hidden_channels,
            hidden_size=hidden_channels * hidden_channels,
        )

        self.classifier = nn.Linear(
            hidden_channels,
            num_classes,
        )

    def forward(self, history, current):

        # Current weights start from learned parameters
        w1 = self.weight1
        w2 = self.weight2

        h1 = torch.zeros(
            1,
            w1.numel(),
            device=w1.device,
        )

        c1 = torch.zeros_like(h1)

        h2 = torch.zeros(
            1,
            w2.numel(),
            device=w2.device,
        )

        c2 = torch.zeros_like(h2)

        # Evolve GCN weights through historical snapshots
        for graph in history:

            # Use graph-level feature summary
            graph_summary = graph.x.mean(dim=0).unsqueeze(0)

            h1, c1 = self.evolve1(
                graph_summary,
                (h1, c1),
            )

            w1 = h1.view(
                self.weight1.shape
            )

            # Hidden representation from current
            x = graph.x @ w1
            x = F.relu(x)

            hidden_summary = x.mean(dim=0).unsqueeze(0)

            h2, c2 = self.evolve2(
                hidden_summary,
                (h2, c2),
            )

            w2 = h2.view(
                self.weight2.shape
            )

        # Apply evolved weights to current graph
        x = current.x @ w1
        x = F.relu(x)

        # Graph convolution using current topology
        row, col = current.edge_index

        messages = x[col]

        aggregated = torch.zeros_like(x)
        aggregated.index_add_(
            0,
            row,
            messages,
        )

        degree = torch.bincount(
            row,
            minlength=current.num_nodes,
        ).float().unsqueeze(1)

        degree = degree.clamp(min=1)

        x = aggregated / degree

        x = x @ w2
        x = F.relu(x)

        logits = self.classifier(x)

        return logits

However, I would actually simplify this before running it. The above is useful conceptually, but manually reproducing GCN normalization makes the implementation unnecessarily messy.

For your exploration repo, I'd rather make the evolving-weight mechanism explicit while using PyG's GCNConv machinery where possible.

More importantly, there is a conceptual point we should preserve:

Since nodes do not persist between snapshots, EvolveGCN should evolve the model parameters, not node embeddings.

That is precisely why it is interesting here.

In [ ]:
model = EvolveGCN(
    in_channels=165,
    hidden_channels=64,
    num_classes=2,
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01,
    weight_decay=5e-4,
)

criterion = nn.CrossEntropyLoss()

I'd stop the temporal-GNN branch here for now.

The important finding is already:

The dataset has temporal snapshots, but a simple temporal representation did not improve over the static graph model.

And you discovered something more fundamental: there are no persistent nodes or cross-timestep edges, so temporal modeling is inherently modeling graph evolution, rather than tracking transaction/node histories.

For your exploratory repo, I'd document EvolveGCN as a potential future experiment, not implement it:

EvolveGCN — not implemented: A natural extension would be to evolve GCN parameters across graph snapshots. This was not pursued because the computational cost was disproportionate to the exploratory objective and the dataset's disconnected snapshot structure makes the expected benefit uncertain.

Then I'd move on to the next modeling paradigm rather than spending more compute trying to improve the GNN.

# IMPORTANT NOTES

1. Scaling: yes, the GNN should probably get scaled features

Your XGBoost experiment can work reasonably well without scaling because trees are largely insensitive to monotonic feature scaling.

A GCN is different. Its message-passing layers perform learned linear transformations of the node features, so very different feature magnitudes can make optimization less well behaved.

For the Elliptic features, I would use the same scaler fitted on the training period only:

from sklearn.preprocessing import StandardScaler

train_nodes = (timesteps <= 34) & (y_graph.numpy() >= 0)

scaler = StandardScaler()

X_train = X_graph[train_nodes].numpy()
scaler.fit(X_train)

X_scaled = scaler.transform(X_graph.numpy())

X_graph = torch.tensor(
    X_scaled,
    dtype=torch.float32,
)

Then build your snapshots from this X_graph.

The important part is:

fit scaler → timesteps 1–34 only
transform → all timesteps

You must not fit the scaler on validation timesteps, because that would let information from the future influence preprocessing.

I'd actually make this common preprocessing for your GNN experiments.

2. And yes: the first GCN's graph split should be corrected

Your current wording is accurate, but I'd make the limitation more explicit.

The current experiment does:

                 ALL 49 TIMESTEPS
                       │
                single graph
                       │
        ┌──────────────┴──────────────┐
        │                             │
    train labels                  val labels
    t = 1–34                      t = 35–49

But the graph itself contains all 49 timesteps.

So even though the loss is only computed on training nodes, message passing can use features from nodes in later snapshots if they are connected by edges.

There is an important correction, though:

You already discovered that every edge connects nodes within the same timestep.

Therefore, in this particular dataset, the concern is actually less severe than the wording suggests.

An edge from a training node at t ≤ 34 cannot reach a future node at t > 34, because:

edge(t_i, t_j)
→ t_i == t_j

So the GCN cannot actually propagate messages from timestep 40 into timestep 20.

However, the validation nodes are still present in the same graph and participate in message passing among themselves. That isn't label leakage, since their features and edges are legitimately available at inference time.

So I would replace your old statement, because it implies a leakage mechanism that your dataset structure rules out.

Better wording

Temporal split and graph structure

The dataset contains one transaction graph with nodes from all 49 timesteps. For the first GCN experiment, we use timesteps 1–34 for training and timesteps 35–49 for validation.

Although the graph is constructed globally, the dataset's edges are strictly intra-timestep: transactions at timestep t are only connected to other transactions at the same timestep. Therefore, message passing from training nodes cannot reach future validation nodes through the graph.

We nevertheless keep this first experiment simple by using a single global graph with temporal masks rather than explicitly constructing separate graph snapshots. A stricter temporal implementation would construct the graph available at each timestep explicitly and evaluate each future snapshot sequentially.

That's much more accurate.

One additional distinction

Your GNN-LSTM experiment actually did address this more explicitly, because you constructed:

G₁, G₂, ..., G₃₄ → training
G₃₅, ..., G₄₉     → validation

and when predicting G_t, its history consisted only of:

G_(t-5), ..., G_(t-1)

So your temporal experiment has a cleaner temporal protocol than the first global-graph GCN.

For the repo, I'd therefore document the GCN as:

Global graph + temporal label masks, with the important dataset-specific fact that edges are intra-timestep, rather than describing it as future-graph leaka

Yes — for the temporal train/validation division itself, there is no problem.

Your split:

Train: timesteps 1–34
Validation: timesteps 35–49

is appropriate and preserves temporal ordering.

The important dataset fact we established is:

Every edge connects two transactions from the same timestep.

So there are no edges crossing from training timesteps into validation timesteps.

Therefore, for the first GCN:

✅ No future timestep can send messages to a training timestep.
✅ Validation labels are not used during training.
✅ The temporal ordering is respected.
⚠️ The graph is constructed globally rather than as separate snapshots, but because edges are strictly intra-timestep, this does not create temporal leakage here.

The only preprocessing issue I'd change is feature scaling, if you want the GCN experiment to have appropriate numerical preprocessing: fit the scaler on timesteps 1–34 and transform all timesteps.

So your methodological description can confidently say:

Temporal split: Transactions from timesteps 1–34 are used for training and transactions from timesteps 35–49 for validation. Because all graph edges are intra-timestep, the graph structure does not connect training transactions to future validation transactions.

# Interesting

If a graph does have edges crossing the temporal split, then the issue becomes more interesting. There are several valid ways to handle it, depending on what you want the model to know at prediction time.

1. Strict temporal graph — safest

For predicting timestep t, only expose graph information available up to t:

Training:
G₁ → G₂ → ... → G₃₄

Validation:
G₃₅ → G₃₆ → ... → G₄₉

For validation at t=35, construct the graph using only information available through timestep 35.

This is the cleanest setup when the real-world question is:

"Given everything known at time t, classify transactions at t."

2. Remove future edges

If you have one global graph but edges have timestamps, construct a training graph containing only:

edge_time <= 34

Then validation can use an appropriate graph containing edges available by the prediction time.

This is useful when you want to retain a single-graph implementation but enforce temporal causality.

3. Snapshot graphs

Construct:

G₁
G₂
...
G₃₄
G₃₅
...
G₄₉

and explicitly control which snapshots are accessible.

This is what you did for the GNN-LSTM and is generally the cleanest representation when the dataset is naturally temporal.

4. Transductive setting — sometimes legitimate

There is another possibility that is easy to confuse with leakage.

Suppose future nodes and their unlabeled features/edges would genuinely be known at prediction time. Then allowing message passing through them can be legitimate:

future node features
        ↓
    message passing
        ↓
current prediction

But you must explicitly define that as the problem setting.

The key distinction is:

Future labels are leakage. Future graph information may or may not be leakage, depending on what would actually be available when making the prediction.

For a fraud/AML-style real-time prediction problem, I'd normally prefer the strict temporal formulation unless the data-generation process justifies transductive access.

In your Elliptic case

You don't need any of these corrections because we measured:

Same timestep edges:       100%
Cross-timestep edges:        0%

So the simple global GCN happens to be temporally safe with respect to graph connectivity. That's actually a useful property to document rather than overcomplicate the first experiment

In [ ]:
# Gogole snapshots
import os
import pandas as pd
import torch
from torch_geometric.data import Data

def load_elliptic_temporal_snapshots(data_dir="../data"):
    """
    Loads Elliptic dataset CSVs and returns a list of PyTorch Geometric Data objects,
    where each item in the list is a self-contained graph snapshot G_t for timestep t.
    """
    # 1. Load CSVs
    classes_df = pd.read_csv(os.path.join(data_dir, "elliptic_txs_classes.csv"))
    edges_df = pd.read_csv(os.path.join(data_dir, "elliptic_txs_edgelist.csv"))
    features_df = pd.read_csv(os.path.join(data_dir, "elliptic_txs_features.csv"), header=None)

    # Clean classes: '1' -> 1 (illicit), '2' -> 0 (licit), 'unknown' -> -1
    classes_df['class'] = classes_df['class'].map({'1': 1, '2': 0, 'unknown': -1})

    # Rename first column of features to txId and second column to time_step
    features_df.rename(columns={0: 'txId', 1: 'time_step'}, inplace=True)
    
    # Merge node information
    nodes_df = pd.merge(features_df, classes_df, on='txId')

    snapshots = []
    
    # 2. Iterate over each unique timestep (1 to 49)
    for t in sorted(nodes_df['time_step'].unique()):
        # Filter nodes belonging strictly to timestep t
        sub_nodes = nodes_df[nodes_df['time_step'] == t].copy()
        
        # Create a local ID mapping for nodes within this snapshot [0, N_t - 1]
        node_id_map = {tx_id: idx for idx, tx_id in enumerate(sub_nodes['txId'])}
        
        # Node Features (columns index 2 to end-1, excluding txId, time_step, and class)
        x = torch.tensor(sub_nodes.iloc[:, 2:-1].values, dtype=torch.float)
        
        # Node Labels (-1 = unknown, 0 = licit, 1 = illicit)
        y = torch.tensor(sub_nodes['class'].values, dtype=torch.long)
        
        # Filter edges where BOTH source and target belong to timestep t
        t_nodes_set = set(sub_nodes['txId'])
        sub_edges = edges_df[
            edges_df['txId1'].isin(t_nodes_set) & 
            edges_df['txId2'].isin(t_nodes_set)
        ]
        
        # Map original txIds to sub-graph local indices [0, N_t - 1]
        src = [node_id_map[tx] for tx in sub_edges['txId1']]
        dst = [node_id_map[tx] for tx in sub_edges['txId2']]
        edge_index = torch.tensor([src, dst], dtype=torch.long)
        
        # 3. Create PyG Data object for Snapshot G_t
        snapshot = Data(x=x, edge_index=edge_index, y=y, time_step=t)
        snapshots.append(snapshot)
        
    return snapshots


def create_sliding_windows(snapshots, window_size=3):
    """
    Groups graph snapshots into sliding window sequences:
    e.g., [G_1, G_2, G_3], [G_2, G_3, G_4], ...
    """
    sequences = []
    for i in range(len(snapshots) - window_size + 1):
        window = snapshots[i : i + window_size]
        sequences.append(window)
    return sequences


# --- Example Usage ---
if __name__ == "__main__":
    # Load all 49 snapshots
    snapshots = load_elliptic_temporal_snapshots(data_dir="../data")
    print(f"Total snapshots loaded: {len(snapshots)}")
    print(f"Snapshot 1 details: {snapshots[0]}")
    
    # Train / Test split by snapshot index
    train_snapshots = snapshots[:34]  # Time steps 1 to 34
    test_snapshots = snapshots[34:]   # Time steps 35 to 49
    
    # Construct sliding window sequences (e.g., window size = 3)
    train_windows = create_sliding_windows(train_snapshots, window_size=3)
    print(f"Total training window sequences: {len(train_windows)}")
    print(f"First sequence contains {len(train_windows[0])} consecutive graph snapshots.")